# Лабораторная работа №3: Foundation models для сегментации: SAM/SAM 2

**Версия:** студенческая  
**Курс:** «Современное компьютерное зрение»

> Сначала выполните notebook в режиме `FAST_DEV_RUN=True`, затем запускайте полный эксперимент. Сохраняйте метрики и checkpoints в `artifacts/`.


## Цели обучения

После работы студент должен уметь:

- Получить маски по point и box prompts.
- Автоматически сформировать prompts из GT или детектора.
- Сравнить zero-shot и refinement head.


## Теоретическая часть

Promptable segmentation оценивает маску $M=f(I,p)$ по изображению и prompt $p$ (точка, box или маска). SAM разделяет тяжёлый image encoder и лёгкий prompt/mask decoder, поэтому один embedding изображения можно использовать для многих запросов.

![SAM architecture](https://raw.githubusercontent.com/facebookresearch/segment-anything/main/assets/model_diagram.png)


### Математическая постановка

Качество promptable-модели оценивают не только IoU, но и числом взаимодействий до достижения целевого качества: $N@IoU\geq\tau$. Для ускорения разметки важна доля масок, принимаемых без существенной коррекции.


### Материалы

- [Segment Anything](https://github.com/facebookresearch/segment-anything)
- [Transformers SAM docs](https://huggingface.co/docs/transformers/model_doc/sam)

Ссылки даны как дополнительный материал. При изменении API сверяйтесь с актуальной официальной документацией и фиксируйте версии зависимостей.


## Практическое задание

1. Получить маски по point и box prompts.
2. Автоматически сформировать prompts из GT или детектора.
3. Сравнить zero-shot и refinement head.

**Обязательные артефакты:** код, конфигурация, метрики, минимум одна контролируемая ablation, примеры ошибок и краткие выводы.


### Профиль вычислений

- **Основной режим:** SAM ViT-B/MobileSAM inference, frozen encoder.
- **Ограничение данных:** до 200 изображений 512 px.
- **Fallback:** CPU на 20 изображениях или MobileSAM.
- Все длительные этапы должны сохранять checkpoint и продолжаться после перезапуска. Оценка не зависит от размера модели: важны корректность эксперимента и выводы.


**Оценивание.** Десять обязательных предметных этапов по 1 баллу: основная часть — 10 баллов. Творческий бонус — отдельно 1 балл, не заменяет обязательные этапы. Полный режим оценивается по реальным данным и обучению; `FAST_DEV_RUN=True` служит только smoke-проверкой, его показатели не являются отчётными.


## Реализация


In [1]:
from pathlib import Path
import json, os, random, time
import numpy as np

SEED = 42
FAST_DEV_RUN = False
ARTIFACTS = Path(os.getenv("CV_ARTIFACTS", "artifacts"))
ARTIFACTS.mkdir(exist_ok=True)
random.seed(SEED); np.random.seed(SEED)

try:
    import torch
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
except ImportError:
    torch = None
    DEVICE = "cpu"

if DEVICE == "cpu":
    torch.set_num_threads(os.cpu_count() or 1)

_CANDIDATES = [
    Path.cwd().parent.parent / "data",
    Path.cwd().parent / "data",
    Path.cwd() / "data",
]
DATA_ROOT = Path(
    os.getenv("CV_COURSE_DATA")
    or next((c for c in _CANDIDATES if (c / "oxford-iiit-pet").exists()), _CANDIDATES[0])
)
DATA_ROOT.mkdir(parents=True, exist_ok=True)
print({"device": DEVICE, "fast_dev_run": FAST_DEV_RUN, "data_root": str(DATA_ROOT)})

{'device': 'cpu', 'fast_dev_run': False, 'data_root': 'd:\\data'}


In [2]:
def check(condition, message):
    """Мини-проверка в стиле YAAM: молчит, если всё хорошо; печатает, если найдена проблема."""
    if condition:
        print(f"✔ {message}")
    else:
        print(f"✘ ВНИМАНИЕ: {message}")

### Загрузка SAM и отложенных масок

*Вес: 1 балл.*

**Постановка.** Сравнение prompts возможно лишь на одних размеченных изображениях.

Выполните следующие действия:
1. Загрузите pretrained SAM и test Oxford-IIIT Pet.
2. Зафиксируйте индексы проверки.

**Результат.** dataset_manifest.json и объекты processor, model, eval_ds.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [3]:
# Реализуйте это задание по постановке выше.
import torch
import numpy as np
from PIL import Image
from torchvision.datasets import OxfordIIITPet
import torchvision.transforms.functional as TF
from transformers import SamModel, SamProcessor

# Загрузка SAM
SAM_CHECKPOINT = "facebook/sam-vit-base"
processor = SamProcessor.from_pretrained(SAM_CHECKPOINT)
sam_model = SamModel.from_pretrained(SAM_CHECKPOINT).to(DEVICE)
sam_model.eval()

# Загрузка test-раздела Oxford-IIIT Pet
raw_test = OxfordIIITPet(
    root=DATA_ROOT, split="test", target_types="segmentation", download=False
)

# Ограничиваем выборку для CPU
eval_indices = list(range(len(raw_test)))
eval_images = []
eval_masks = []

for idx in eval_indices:
    img, mask = raw_test[idx]
    img = TF.resize(img, (256, 256), interpolation=TF.InterpolationMode.BILINEAR)
    mask = TF.resize(mask, (256, 256), interpolation=TF.InterpolationMode.NEAREST)
    eval_images.append(np.array(img))
    eval_masks.append(np.array(mask))

# Фиксация манифеста
manifest = {"indices": eval_indices, "n_images": len(eval_indices)}
with open(ARTIFACTS / "dataset_manifest.json", "w", encoding="utf-8") as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)

eval_ds = list(zip(eval_images, eval_masks))
print(f"✔ Загружено {len(eval_ds)} test-изображений; SAM готов")

c:\Users\thras\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 314/314 [00:00<00:00, 711.84it/s]


✔ Загружено 3669 test-изображений; SAM готов


### Геометрия подсказок и IoU

*Вес: 1 балл.*

**Постановка.** Box и point должны воспроизводимо извлекаться из одной истинной маски.

Выполните следующие действия:
1. Реализуйте IoU масок и ограничивающий прямоугольник.
2. Документируйте, что подсказки из GT моделируют действия разметчика.

**Результат.** Функции iou и box_from_mask.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [4]:
# Реализуйте это задание по постановке выше.
def iou(pred_mask, gt_mask):
    """IoU двух бинарных масок (bool или 0/1)."""
    pred = pred_mask.astype(bool)
    gt = gt_mask.astype(bool)
    inter = (pred & gt).sum()
    union = (pred | gt).sum()
    return inter / union if union > 0 else 0.0

def box_from_mask(mask):
    """Bounding box [x_min, y_min, x_max, y_max] из бинарной маски."""
    ys, xs = np.where(mask > 0)
    if len(xs) == 0:
        return None
    return [int(xs.min()), int(ys.min()), int(xs.max()), int(ys.max())]

### Предсказание SAM по подсказке

*Вес: 1 балл.*

**Постановка.** Encoder и mask decoder должны получать корректный тип prompt.

Выполните следующие действия:
1. Подайте point или box в processor.
2. Выберите маску по предсказанной SAM оценке IoU.

**Результат.** Функция predict_sam с бинарной маской исходного размера.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [5]:
# Реализуйте это задание по постановке выше.
def predict_sam(image_np, point=None, box=None):
    """
    point: [x, y] или None
    box:   [x_min, y_min, x_max, y_max] или None
    Возвращает бинарную маску исходного размера (bool, H×W).
    """
    kwargs = {"return_tensors": "pt"}
    if point is not None:
        kwargs["input_points"] = [[point]]      # [batch=1, point_batch=1, n_points=1, 2]
    if box is not None:
        kwargs["input_boxes"] = [[box]]         # [batch=1, n_boxes=1, 4]

    inputs = processor(image_np, **kwargs).to(DEVICE)

    with torch.inference_mode():
        outputs = sam_model(**inputs)

    # pred_masks: [1, point_batch, num_masks, H_low, W_low]
    # post_process_masks -> list длиной batch_size,
    # каждый элемент формы [point_batch, num_masks, H, W]
    masks = processor.image_processor.post_process_masks(
        outputs.pred_masks.cpu(),
        inputs["original_sizes"].cpu(),
        inputs["reshaped_input_sizes"].cpu(),
    )[0]

    # Убираем лишнюю размерность point_batch, если она есть
    if masks.ndim == 4:
        # [point_batch, num_masks, H, W] -> [num_masks, H, W]
        masks = masks.squeeze(0)
    elif masks.ndim == 3:
        # уже [num_masks, H, W]
        pass
    else:
        raise RuntimeError(f"Неожиданная форма масок: {tuple(masks.shape)}")

    # iou_scores: [1, point_batch, num_masks] -> плоский массив
    scores = outputs.iou_scores.cpu().numpy().flatten()   # длина = num_masks

    best_idx = int(np.argmax(scores))
    best_mask = masks[best_idx].numpy().squeeze() > 0.0   # bool [H, W]
    return best_mask

### Сравнение point и box

*Вес: 1 балл.*

**Постановка.** Степень интерактивной подсказки влияет на качество сегментации.

Выполните следующие действия:
1. Получите две маски для каждого отложенного изображения.
2. Сохраните IoU, площади и координаты prompts.

**Результат.** prompt_results.json с реальными оценками SAM.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [6]:
# Реализуйте это задание по постановке выше.
results = []

for idx, (img_np, gt_mask) in enumerate(eval_ds[:30]):
    # GT-маска питомца: foreground = 1, boundary = 3 -> объединяем
    fg_mask = np.isin(gt_mask, [1, 3])

    if fg_mask.sum() == 0:
        continue

    box = box_from_mask(fg_mask)
    if box is None:
        continue

    # Центр маски как point
    ys, xs = np.where(fg_mask)
    point = [int(xs.mean()), int(ys.mean())]

    pred_point = predict_sam(img_np, point=point)
    pred_box = predict_sam(img_np, box=box)

    results.append({
        "index": idx,
        "iou_point": float(iou(pred_point, fg_mask)),
        "iou_box": float(iou(pred_box, fg_mask)),
        "area_gt": int(fg_mask.sum()),
        "point": point,
        "box": box,
    })

with open(ARTIFACTS / "prompt_results.json", "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

mean_iou_point = np.mean([r["iou_point"] for r in results])
mean_iou_box = np.mean([r["iou_box"] for r in results])
print(f"✔ Point IoU: {mean_iou_point:.4f} | Box IoU: {mean_iou_box:.4f}")

✔ Point IoU: 0.6068 | Box IoU: 0.7179


### Обучение supervised baseline

*Вес: 1 балл.*

**Постановка.** Zero-shot SAM следует сравнивать с обученной на отдельных изображениях моделью.

Выполните следующие действия:
1. Подготовьте бинарные маски trainval.
2. Обучите LRASPP без доступа к test и сохраните веса.

**Результат.** supervised.pt с обученными весами.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [7]:
# Реализуйте это задание по постановке выше.
import torchvision.models.segmentation as seg_models
import torch.nn as nn
from torch.utils.data import DataLoader, random_split, Subset

# Загружаем trainval, конвертируем в бинарную маску (pet=1, фон/контур=0)
raw_train = OxfordIIITPet(
    root=DATA_ROOT, split="trainval", target_types="segmentation", download=False
)

MAX_TRAIN = 64 if FAST_DEV_RUN else 200
train_subset = Subset(raw_train, list(range(min(MAX_TRAIN, len(raw_train)))))

class BinaryPetDataset(torch.utils.data.Dataset):
    def __init__(self, subset):
        self.subset = subset
    def __len__(self):
        return len(self.subset)
    def __getitem__(self, idx):
        img, mask = self.subset[idx]
        img = TF.resize(img, (256, 256), interpolation=TF.InterpolationMode.BILINEAR)
        mask = TF.resize(mask, (256, 256), interpolation=TF.InterpolationMode.NEAREST)
        img_t = TF.to_tensor(img)
        img_t = TF.normalize(img_t, [0.485,0.456,0.406], [0.229,0.224,0.225])
        m = torch.from_numpy(np.array(mask)).long()
        m = torch.clamp(m - 1, 0, 2)  # 0: pet, 1: background, 2: boundary
        binary = (m == 0).long()      # бинарная маска: 1 = pet
        return img_t, binary

bin_train = BinaryPetDataset(train_subset)

val_size = max(1, int(0.15 * len(bin_train)))
train_size = len(bin_train) - val_size
train_ds, val_ds = random_split(bin_train, [train_size, val_size],
                                generator=torch.Generator().manual_seed(SEED))

batch_size = 4 if FAST_DEV_RUN else 8
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=0, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=0)

# LRASPP с двумя классами (фон / питомец)
sup_model = seg_models.lraspp_mobilenet_v3_large(
    weights=seg_models.LRASPP_MobileNet_V3_Large_Weights.DEFAULT
)
sup_model.classifier.high_classifier = nn.Conv2d(128, 2, kernel_size=1)
sup_model.classifier.low_classifier = nn.Conv2d(40, 2, kernel_size=1)
sup_model.to(DEVICE)

optimizer = torch.optim.AdamW(sup_model.parameters(), lr=1e-3, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()

def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0
    total_samples = 0
    for imgs, masks in loader:
        imgs, masks = imgs.to(device), masks.to(device)
        optimizer.zero_grad()
        out = model(imgs)["out"]
        loss = criterion(out, masks)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * imgs.size(0)
        total_samples += imgs.size(0)
    return total_loss / max(1, total_samples)

epochs = 2 if FAST_DEV_RUN else 4
history = []
for ep in range(epochs):
    l = train_one_epoch(sup_model, train_loader, optimizer, criterion, DEVICE)
    history.append(l)
    print(f"Epoch {ep+1}/{epochs} | loss={l:.4f}")

torch.save(sup_model.state_dict(), ARTIFACTS / "supervised.pt")
print("✔ supervised.pt сохранён")

Epoch 1/4 | loss=0.2814
Epoch 2/4 | loss=0.1336
Epoch 3/4 | loss=0.1171
Epoch 4/4 | loss=0.1004
✔ supervised.pt сохранён


### Парное сравнение с baseline

*Вес: 1 балл.*

**Постановка.** Парные изображения исключают разницу в составе оценочной выборки.

Выполните следующие действия:
1. Предскажите test-маски supervised модели.
2. Сравните её IoU с SAM box на тех же индексах.

**Результат.** supervised_comparison.json с парными IoU.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [8]:
# Реализуйте это задание по постановке выше.
sup_model.eval()
comparison = []

for idx, (img_np, gt_mask) in enumerate(eval_ds):
    fg_mask = np.isin(gt_mask, [1, 3])
    if fg_mask.sum() == 0:
        continue

    # IoU SAM box (из предыдущего этапа)
    box_entry = next((r for r in results if r["index"] == idx), None)
    iou_sam_box = box_entry["iou_box"] if box_entry else 0.0

    # Предсказание supervised
    img_t = TF.to_tensor(img_np)
    img_t = TF.normalize(img_t, [0.485,0.456,0.406], [0.229,0.224,0.225])
    with torch.inference_mode():
        out = sup_model(img_t.unsqueeze(0).to(DEVICE))["out"]
        pred_bin = out.argmax(dim=1).squeeze().cpu().numpy().astype(bool)

    comparison.append({
        "index": idx,
        "iou_sam_box": float(iou_sam_box),
        "iou_supervised": float(iou(pred_bin, fg_mask)),
    })

with open(ARTIFACTS / "supervised_comparison.json", "w", encoding="utf-8") as f:
    json.dump(comparison, f, ensure_ascii=False, indent=2)

print(f"✔ SAM box: {np.mean([c['iou_sam_box'] for c in comparison]):.4f} | "
      f"Supervised: {np.mean([c['iou_supervised'] for c in comparison]):.4f}")

✔ SAM box: 0.0059 | Supervised: 0.6595


### Обучающие маски для корректора

*Вес: 1 балл.*

**Постановка.** Корректор должен обучаться только на train-изображениях и масках SAM.

Выполните следующие действия:
1. Получите SAM-предсказания для trainval.
2. Сформируйте четырёхканальные входы из RGB и маски.

**Результат.** refine_train из реальных изображений и масок.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [9]:
# Реализуйте это задание по постановке выше.
# Для корректора используем SAM-маски на train-изображениях (не test)
refine_inputs = []
MAX_REFINE = 32 if FAST_DEV_RUN else 80
refine_subset = Subset(raw_train, list(range(min(MAX_REFINE, len(raw_train)))))

for idx in range(len(refine_subset)):
    img_pil, mask_pil = refine_subset[idx]
    img_np = np.array(TF.resize(img_pil, (256, 256), interpolation=TF.InterpolationMode.BILINEAR))
    gt_mask = np.array(TF.resize(mask_pil, (256, 256), interpolation=TF.InterpolationMode.NEAREST))
    fg = np.isin(gt_mask, [1, 3])
    if fg.sum() == 0:
        continue
    box = box_from_mask(fg)
    if box is None:
        continue
    sam_mask = predict_sam(img_np, box=box)  # бинарная маска SAM

    # 4-канальный вход: RGB + маска SAM
    img_t = TF.to_tensor(img_np)
    img_t = TF.normalize(img_t, [0.485,0.456,0.406], [0.229,0.224,0.225])
    mask_t = torch.from_numpy(sam_mask.astype(np.float32)).unsqueeze(0)
    input_4ch = torch.cat([img_t, mask_t], dim=0)  # [4, 256, 256]

    refine_inputs.append((input_4ch, torch.from_numpy(fg.astype(np.float32))))

refine_train = refine_inputs
print(f"✔ Сформировано {len(refine_train)} обучающих примеров для корректора")

✔ Сформировано 80 обучающих примеров для корректора


### Обучение корректирующей головы

*Вес: 1 балл.*

**Постановка.** Обучаемая свёрточная голова может уточнять ошибочную маску.

Выполните следующие действия:
1. Обучите голову на подготовленных четырёхканальных входах.
2. Сохраните веса для повторного использования.

**Результат.** refinement.pt после фактической оптимизации.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [10]:
# Реализуйте это задание по постановке выше.
import torch.nn as nn

class RefinementHead(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(4, 32, 3, padding=1), nn.ReLU(),
            nn.Conv2d(32, 32, 3, padding=1), nn.ReLU(),
            nn.Conv2d(32, 1, 1),
        )
    def forward(self, x):
        return self.net(x)  # логиты [B,1,H,W]

refine_model = RefinementHead().to(DEVICE)
refine_optim = torch.optim.AdamW(refine_model.parameters(), lr=1e-3)
bce = nn.BCEWithLogitsLoss()

refine_loader = DataLoader(
    refine_train, batch_size=4 if FAST_DEV_RUN else 8,
    shuffle=True, num_workers=0, drop_last=True
)

refine_epochs = 3 if FAST_DEV_RUN else 6
for ep in range(refine_epochs):
    refine_model.train()
    total = 0.0
    n = 0
    for x, y in refine_loader:
        x, y = x.to(DEVICE), y.to(DEVICE).unsqueeze(1)
        refine_optim.zero_grad()
        logits = refine_model(x)
        loss = bce(logits, y)
        loss.backward()
        refine_optim.step()
        total += loss.item() * x.size(0)
        n += x.size(0)
    print(f"Refine epoch {ep+1}/{refine_epochs} | loss={total/max(1,n):.4f}")

torch.save(refine_model.state_dict(), ARTIFACTS / "refinement.pt")
print("✔ refinement.pt сохранён")

Refine epoch 1/6 | loss=0.6330
Refine epoch 2/6 | loss=0.5321
Refine epoch 3/6 | loss=0.4246
Refine epoch 4/6 | loss=0.3662
Refine epoch 5/6 | loss=0.3531
Refine epoch 6/6 | loss=0.3448
✔ refinement.pt сохранён


### Независимая оценка коррекции

*Вес: 1 балл.*

**Постановка.** Качество коррекции необходимо измерить на отложенных изображениях.

Выполните следующие действия:
1. Примените обученную голову к SAM-маскам test.
2. Сохраните IoU и долю исходных масок, требующих существенной правки.

**Результат.** adapted_comparison.json с измеренными IoU.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [12]:
# Реализуйте это задание по постановке выше.
refine_model.eval()
adapted = []

for idx, (img_np, gt_mask) in enumerate(eval_ds[:30]):
    fg = np.isin(gt_mask, [1, 3])
    if fg.sum() == 0:
        continue
    box = box_from_mask(fg)
    if box is None:
        continue

    sam_mask = predict_sam(img_np, box=box)
    iou_sam = iou(sam_mask, fg)

    # Коррекция
    img_t = TF.to_tensor(img_np)
    img_t = TF.normalize(img_t, [0.485,0.456,0.406], [0.229,0.224,0.225])
    mask_t = torch.from_numpy(sam_mask.astype(np.float32)).unsqueeze(0)
    inp = torch.cat([img_t, mask_t], dim=0).unsqueeze(0).to(DEVICE)

    with torch.inference_mode():
        logit = refine_model(inp)
        corrected = (torch.sigmoid(logit).squeeze().cpu().numpy() > 0.5)

    iou_corr = iou(corrected, fg)

    adapted.append({
        "index": idx,
        "iou_sam": float(iou_sam),
        "iou_corrected": float(iou_corr),
        "need_edit": bool(iou_sam < 0.7),  # доля масок, требовавших правки
    })

with open(ARTIFACTS / "adapted_comparison.json", "w", encoding="utf-8") as f:
    json.dump(adapted, f, ensure_ascii=False, indent=2)

mean_sam = np.mean([a["iou_sam"] for a in adapted])
mean_corr = np.mean([a["iou_corrected"] for a in adapted])
need_edit_ratio = np.mean([a["need_edit"] for a in adapted])
print(f"✔ SAM: {mean_sam:.4f} → Corrected: {mean_corr:.4f} | "
      f"доля требующих правки: {need_edit_ratio:.2%}")

✔ SAM: 0.7179 → Corrected: 0.7255 | доля требующих правки: 40.00%


### Сопоставление трудных случаев

*Вес: 1 балл.*

**Постановка.** Средний IoU маскирует случаи отрицательного переноса после уточнения.

Выполните следующие действия:
1. Соедините SAM, supervised и corrected IoU по test-индексам.
2. Укажите изображения, где корректор снизил IoU по сравнению с SAM.

**Результат.** correction_failures.json с реальными индексами и тремя IoU.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [13]:
# Реализуйте это задание по постановке выше.
failures = [a for a in adapted if a["iou_corrected"] < a["iou_sam"]]
failures.sort(key=lambda x: x["iou_corrected"] - x["iou_sam"])

with open(ARTIFACTS / "correction_failures.json", "w", encoding="utf-8") as f:
    json.dump(failures, f, ensure_ascii=False, indent=2)

print(f"✔ Найдено {len(failures)} случаев, где коррекция снизила IoU")

✔ Найдено 3 случаев, где коррекция снизила IoU


### Творческий бонус: второй тип точки

*Вес: 1 дополнительный балл.*

**Постановка.** Качество SAM зависит от выбора положительной точки.

Выполните следующие действия:
1. Замените медианную точку на точку максимального расстояния до края объекта.
2. На тех же test-изображениях сравните IoU с исходной подсказкой.

**Результат.** bonus_points.json с парными IoU для двух стратегий.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [14]:
# Реализуйте это задание по постановке выше.
def farthest_point_from_edge(mask):
    """Точка внутри маски с максимальным расстоянием до фона (приближённо)."""
    from scipy.ndimage import distance_transform_edt
    dist = distance_transform_edt(mask > 0)
    ys, xs = np.where(dist == dist.max())
    return [int(xs[0]), int(ys[0])]

# Для простоты: евклидово расстояние до границы через бинарную эрозию
def max_inscribed_point(mask):
    m = mask.astype(np.uint8)
    from scipy.ndimage import binary_erosion
    current = m.copy()
    while current.sum() > 0:
        prev = current.copy()
        current = binary_erosion(current, iterations=1)
        if current.sum() == 0:
            ys, xs = np.where(prev)
            break
    return [int(xs[len(xs)//2]), int(ys[len(ys)//2])]

bonus = []
for idx, (img_np, gt_mask) in enumerate(eval_ds[:30]):
    fg = np.isin(gt_mask, [1, 3])
    if fg.sum() == 0:
        continue

    ys, xs = np.where(fg)
    median_point = [int(xs.mean()), int(ys.mean())]
    deep_point = max_inscribed_point(fg)

    iou_median = iou(predict_sam(img_np, point=median_point), fg)
    iou_deep = iou(predict_sam(img_np, point=deep_point), fg)

    bonus.append({
        "index": idx,
        "iou_median_point": float(iou_median),
        "iou_deep_point": float(iou_deep),
        "median_point": median_point,
        "deep_point": deep_point,
    })

with open(ARTIFACTS / "bonus_points.json", "w", encoding="utf-8") as f:
    json.dump(bonus, f, ensure_ascii=False, indent=2)

print(f"✔ Медианная точка: {np.mean([b['iou_median_point'] for b in bonus]):.4f} | "
      f"Глубинная точка: {np.mean([b['iou_deep_point'] for b in bonus]):.4f}")

✔ Медианная точка: 0.6068 | Глубинная точка: 0.6174


## Анализ результатов

Сопоставьте измеренные показатели на отложенных реальных данных. Укажите бюджет, ошибки модели и ограничения сокращённого запуска.


## Выводы

Сформулируйте предметный вывод по сохранённым артефактам.
